# GroupChat Orchestration with Claude and AG2

This notebook demonstrates AG2's GroupChat feature — orchestrating multiple specialized Claude agents that collaborate to solve complex tasks.

## What you'll learn
- How to create multiple specialized agents with different system prompts
- How to set up a GroupChat with automatic speaker selection
- How Claude agents collaborate in a managed conversation

In [1]:
%pip install "ag2[anthropic]>=0.11.4,<1.0" -q

/Users/faridunm/Documents/WORK/AG2/Opensource/claude-cookbooks/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from autogen import AssistantAgent, GroupChat, GroupChatManager, LLMConfig, UserProxyAgent

llm_config = LLMConfig(
    {
        "model": "claude-sonnet-4-6",
        "api_key": os.environ.get("ANTHROPIC_API_KEY"),
        "api_type": "anthropic",
    }
)

## Creating Specialized Agents

We'll create a team of three agents, each with a distinct role:
- **Researcher**: Gathers and analyzes information
- **Writer**: Creates polished content from research
- **Reviewer**: Provides constructive feedback

In [3]:
researcher = AssistantAgent(
    name="Researcher",
    system_message=(
        "You are a research specialist. Your job is to gather key facts, data points, "
        "and insights on the given topic. Present your findings in a structured format. "
        "Focus on accuracy and completeness."
    ),
    llm_config=llm_config,
)

writer = AssistantAgent(
    name="Writer",
    system_message=(
        "You are a skilled content writer. Take the research provided and create "
        "a well-structured, engaging summary. Focus on clarity and readability. "
        "Keep the output concise \u2014 no more than 200 words."
    ),
    llm_config=llm_config,
)

reviewer = AssistantAgent(
    name="Reviewer",
    system_message=(
        "You are a quality reviewer. Evaluate the written content for accuracy, "
        "clarity, and completeness. Provide specific, actionable feedback. "
        "If the content is satisfactory, reply with TERMINATE."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

## Setting Up GroupChat

GroupChat manages the conversation flow between agents. The GroupChatManager uses Claude to decide which agent should speak next based on the conversation context.

In [4]:
group_chat = GroupChat(
    agents=[user_proxy, researcher, writer, reviewer],
    messages=[],
    max_round=8,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

In [5]:
chat = user_proxy.run(
    manager,
    message="Create a brief overview of how large language models are being used in healthcare.",
)
chat.process()

User (to chat_manager):



Create a brief overview of how large language models are being used in healthcare.



--------------------------------------------------------------------------------



Next speaker: Researcher



/Users/faridunm/Documents/WORK/AG2/Opensource/claude-cookbooks/.venv/lib/python3.11/site-packages/autogen/oai/anthropic.py:1609: UserWarning: Cost calculation not available for model claude-sonnet-4-6
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)


...


Next speaker: Reviewer



Reviewer (to chat_manager):



I'm not able to continue without a specific direction from you. The review process is complete and the document has been given a final verdict of **publication-ready**.

To move forward, please clarify what you would like me to do. For example:

- **"Please apply the final phrasing suggestion to produce a clean final draft"**
- **"Please expand this into a full article"**
- **"Please review this new piece of content"**

Without a clear instruction, continuing would mean generating unnecessary content, which would not serve you well.

TERMINATE



--------------------------------------------------------------------------------



>>>>>>>> TERMINATING RUN (d046f732-48f2-4ac2-b141-040a4a81dc12): Maximum rounds (8) reached


## Understanding the Flow

The GroupChatManager orchestrates the conversation:
1. **User** provides the initial task
2. **Researcher** gathers key information about the topic
3. **Writer** synthesizes the research into polished content
4. **Reviewer** evaluates and either requests changes or approves

This pattern is powerful for any workflow requiring multiple perspectives — content creation, code review, analysis pipelines, and more.

## Next Steps

- Try adding more specialized agents (e.g., Fact-Checker, Editor)
- Experiment with `speaker_selection_method="round_robin"` for predictable turn order
- Explore [AG2's advanced GroupChat patterns](https://docs.ag2.ai/docs/tutorial/conversation-patterns) for complex workflows
- See the [AG2 Documentation](https://docs.ag2.ai) for the full feature set